In [ ]:
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FileUpload
from IPython.display import clear_output, display, HTML

# ------------------------------
# Header / Styling
# ------------------------------

display(HTML("""
<h1 style='text-align:center; font-family:Arial; color:#333;'>📡 2D Fourier Transform Explorer</h1>
<p style='text-align:center; font-size:15px; color:#555;'>
Upload an image, inspect its Fourier spectrum, and apply a low-pass filter interactively.
</p>
<hr style='margin:15px 0px;'>
"""))

# Collapsible Instructions
display(HTML("""
<details style="font-family:Arial;">
  <summary style="font-size:16px; cursor:pointer;">ℹ️ Instructions</summary>
  <p style="font-size:14px; margin-top:10px;">
  • Upload any JPG/PNG image.<br>
  • Images larger than 512×512 are automatically resized while keeping aspect ratio.<br>
  • Adjust the “Low-pass filter radius” slider to remove high frequencies.<br>
  • Works on mobile devices.<br>
  Best results: grayscale or simple images.
  </p>
</details>
<br>
"""))

# ------------------------------
# Helper functions
# ------------------------------

def get_uploaded_file_content(upload_widget):
    """Safely extract bytes from the FileUpload widget in Voila."""
    if not upload_widget.value:
        return None
    val = upload_widget.value
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    if isinstance(val, tuple):
        return val[0]["content"]
    return None


MAX_SIZE = (512, 512)

def load_and_resize_image(content_bytes):
    """Load image, resize if needed, preserve aspect ratio."""
    img = Image.open(io.BytesIO(content_bytes))

    original_size = img.size
    resized = False

    if img.width > MAX_SIZE[0] or img.height > MAX_SIZE[1]:
        img.thumbnail(MAX_SIZE)
        resized = True

    return img, np.array(img), resized, original_size, img.size


# ------------------------------
# Widgets
# ------------------------------

upload = FileUpload(
    accept="image/*",
    description="📤 Upload Image",
    multiple=False,
    layout=widgets.Layout(width="200px")
)

cutoff_slider = widgets.IntSlider(
    value=20, min=1, max=150, step=1,
    description="Filter radius:",
    style={'description_width': 'initial'},
    continuous_update=True,
    layout=widgets.Layout(width="300px")
)

status = widgets.Output()
out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()

# ------------------------------
# Update function
# ------------------------------

def update_plot(change=None):
    status.clear_output()
    out_original.clear_output()
    out_fft.clear_output()
    out_filtered.clear_output()

    content = get_uploaded_file_content(upload)
    if content is None:
        with status:
            print("Waiting for image…")
        return

    try:
        with status:
            print("Loading image...")

        pil_img, img_array, resized, orig_size, new_size = load_and_resize_image(content)

        with status:
            if resized:
                print(f"Image was large ({orig_size}), resized to {new_size}")
            else:
                print(f"Image size: {orig_size} (no resize needed)")
            print("Computing FFT…")

        # --- Show original ---
        with out_original:
            plt.figure(figsize=(3.8, 3.8))
            plt.imshow(pil_img, cmap="gray")
            plt.title("Original Image")
            plt.axis("off")
            plt.show()

        # --- FFT ---
        f = np.fft.fft2(img_array)
        fshift = np.fft.fftshift(f)
        magnitude = np.log1p(np.abs(fshift))

        with out_fft:
            plt.figure(figsize=(3.8, 3.8))
            plt.imshow(magnitude, cmap="magma")
            plt.title("FFT Magnitude")
            plt.axis("off")
            plt.show()

        # --- Low-pass filter ---
        rows, cols = img_array.shape[:2]
        crow, ccol = rows//2, cols//2
        cutoff = cutoff_slider.value

        mask = np.zeros((rows, cols))
        mask[crow-cutoff:crow+cutoff, ccol-cutoff:ccol+cutoff] = 1

        fshift_filtered = fshift * mask
        f_ishift = np.fft.ifftshift(fshift_filtered)
        filtered_img = np.abs(np.fft.ifft2(f_ishift))

        with out_filtered:
            plt.figure(figsize=(3.8, 3.8))
            plt.imshow(filtered_img, cmap="gray")
            plt.title("Low-pass Filter Result")
            plt.axis("off")
            plt.show()

    except Exception as e:
        with status:
            print("Error:", e)


# ------------------------------
# Callbacks
# ------------------------------

upload.observe(update_plot, names='value')
cutoff_slider.observe(update_plot, names='value')

# ------------------------------
# Layout
# ------------------------------

ui = widgets.VBox([
    widgets.HBox([upload, cutoff_slider]),
    status,
    widgets.HBox([
        out_original,
        out_fft,
        out_filtered
    ])
])

display(ui)
